# 01 · Exploration de xView

**Objectif.** Prendre en main le jeu xView : nombre d'images, format des annotations GeoJSON, densité d'objets par image.

**Entrées.** Jeu xView (copie Kaggle `hassanmojab/xview-dataset`) : `train_images/` (846 images GeoTIFF) et `train_labels/xView_train.geojson`.

**Sorties.** Aucune donnée écrite. Constats : 601 937 boîtes annotées ; l'image la plus dense (1831.tif) en compte 7 618.
Figures exportées : `results/figures/exploration_xview_exemples.jpg`, `results/figures/exploration_xview_1831_annotations.jpg`.

*Exécuté sur Kaggle ; les images ont été retirées des sorties pour alléger le notebook.*

In [ ]:
# --- Configuration des chemins ---
# Exécution d'origine sur Kaggle : KAGGLE_INPUT = /kaggle/input/datasets, WORK_DIR = /kaggle/working.
# En local, définir les variables d'environnement GOD_EYE_INPUT et GOD_EYE_WORK (ou modifier les valeurs ci-dessous).
import os
KAGGLE_INPUT = os.environ.get("GOD_EYE_INPUT", "/kaggle/input/datasets")
WORK_DIR = os.environ.get("GOD_EYE_WORK", "/kaggle/working")

In [3]:
import os
print(os.listdir(KAGGLE_INPUT))


['datasets']


In [6]:
XVIEW_ROOT = f"{KAGGLE_INPUT}/hassanmojab/xview-dataset"
IMAGES_DIR = os.path.join(XVIEW_ROOT, "train_images", "train_images")

all_images = sorted(os.listdir(IMAGES_DIR))
print(f"{len(all_images)} images trouvées")
print("Exemples de noms :", all_images[:5])

846 images trouvées
Exemples de noms : ['10.tif', '100.tif', '102.tif', '1036.tif', '1037.tif']


In [8]:
from PIL import Image
import matplotlib.pyplot as plt

In [9]:
n = 6
fig, axes = plt.subplots(2, 3, figsize=(15, 10))

for ax, img_name in zip(axes.flat, all_images[:n]):
    img_path = os.path.join(IMAGES_DIR, img_name)
    img = Image.open(img_path)
    ax.imshow(img)
    ax.set_title(img_name, fontsize=9)
    ax.axis("off")

plt.tight_layout()
plt.show()

In [10]:
import json

GEOJSON_PATH = f"{KAGGLE_INPUT}/hassanmojab/xview-dataset/train_labels/xView_train.geojson"

with open(GEOJSON_PATH) as f:
    geo = json.load(f)

print(geo.keys())
print("Nombre total de features (objets annotés) :", len(geo["features"]))
print("Exemple de feature :")
print(json.dumps(geo["features"][0], indent=2))

dict_keys(['crs', 'type', 'features', 'name'])
Nombre total de features (objets annotés) : 601937
Exemple de feature :
{
  "type": "Feature",
  "properties": {
    "bounds_imcoords": "2712,1145,2746,1177",
    "edited_by": "wwoscarbecerril",
    "cat_id": "1040010028371A00",
    "type_id": 73,
    "ingest_time": "2017/07/24 12:49:09.118+00",
    "index_right": 2356,
    "image_id": "2355.tif",
    "point_geom": "0101000020E6100000616E4E6406A256C03BE6ADA0D6212D40",
    "feature_id": 374410,
    "grid_file": "Grid2.shp"
  },
  "geometry": {
    "type": "Polygon",
    "coordinates": [
      [
        [
          -90.53169885094464,
          14.56603647302396
        ],
        [
          -90.53169885094464,
          14.56614473506768
        ],
        [
          -90.53158140073565,
          14.56614473506768
        ],
        [
          -90.53158140073565,
          14.56603647302396
        ],
        [
          -90.53169885094464,
          14.56603647302396
        ]
      ]
 

In [11]:
import pandas as pd

rows = []
for feat in geo["features"]:
    p = feat["properties"]
    rows.append({
        "image_id": p.get("image_id"),
        "type_id": p.get("type_id"),
        "bounds_imcoords": p.get("bounds_imcoords"),
    })

df = pd.DataFrame(rows)
print(df.shape)
df.head()

(601937, 3)


   image_id  type_id      bounds_imcoords
0  2355.tif       73  2712,1145,2746,1177
1  2355.tif       73  2720,2233,2760,2288
2  2355.tif       73  2687,1338,2740,1399
3  2355.tif       73  2691,1201,2730,1268
4  2355.tif       73    2671,838,2714,869

In [12]:
counts = df["image_id"].value_counts()
print(counts.head(10))

image_id
1831.tif    7618
1106.tif    5750
1080.tif    5736
1217.tif    5609
1109.tif    5346
548.tif     4968
523.tif     4924
600.tif     4605
1216.tif    4550
1465.tif    4509
Name: count, dtype: int64


In [13]:
XVIEW_IMAGES_DIR = f"{KAGGLE_INPUT}/hassanmojab/xview-dataset/train_images/train_images"

image_id = counts.index[0]  # prend l'image la plus riche en objets
boxes_df = df[df["image_id"] == image_id]
print(f"{len(boxes_df)} boîtes pour l'image {image_id}")
boxes_df.head()

7618 boîtes pour l'image 1831.tif


        image_id  type_id    bounds_imcoords
396634  1831.tif       18    71,1847,79,1859
396635  1831.tif       18    50,1766,58,1780
396636  1831.tif       18  178,1567,185,1581
396637  1831.tif       18  596,1590,609,1598
396638  1831.tif       18  780,1597,801,1606

In [14]:
import os
from PIL import Image, ImageDraw
import matplotlib.pyplot as plt

img_path = os.path.join(XVIEW_IMAGES_DIR, image_id)
img = Image.open(img_path).convert("RGB")
draw = ImageDraw.Draw(img)

for _, row in boxes_df.iterrows():
    coords = row["bounds_imcoords"]
    if coords:
        xmin, ymin, xmax, ymax = map(int, coords.split(","))
        draw.rectangle([xmin, ymin, xmax, ymax], outline="red", width=3)

plt.figure(figsize=(12, 12))
plt.imshow(img)
plt.axis("off")
plt.title(f"{image_id} — {len(boxes_df)} objets")
plt.show()